# SkinFLNet++ — publication figures (local or Colab)

**Figures omit matplotlib titles** (journal style: caption under the figure). Use axis labels plus export filenames or `comparison_group_figure_title` / `get_clean_run_name` for prose.

**Where outputs go (shared convention with the ISIC 2018 viz notebook):** in **Setup**, `EXPERIMENT_KEY` is the single switch. Training metrics are read from `PROJECT_ROOT / "results" / EXPERIMENT_KEY`, and journal SVG/PDF exports are written to `PROJECT_ROOT / "figures" / "journal" / EXPERIMENT_KEY` (`JOURNAL_EXPORT_DIR`). This notebook sets `EXPERIMENT_KEY = "isic2019"`.

**Exports:** figures are saved as **SVG and PDF** under `JOURNAL_EXPORT_DIR`, using serif fonts (LaTeX-friendly), journal-sized axes, thick lines, and Seaborn's **`colorblind`** palette for multi-run curves.

**Cross-run comparisons:** for each group, exports include **macro-F1, accuracy, and loss** as separate overlays plus a **stacked three-panel** PDF (`…__panel_macro_acc_loss`) and bar charts at best/last round. Legends use **context-aware names** via `get_clean_run_name` (raw folder names never appear). Baseline `isic2019_dirichlet` is labeled by group (e.g. FedAvg baseline for strategy, α labels for Dirichlet ablations).

**Early stopping:** when comparing runs of different lengths, the **solid** trace is the logged trajectory; the **dashed** tail forward-fills the last metric to the longest run's horizon (lower alpha). **Stars** mark the last round when early stopping fired (see `stopped` / `experiment` in `summary.json`).

**Local-epoch ablations:** call `compare_group("localepochs")` like other groups (x-axis **Round**, same federated horizon as `num_rounds`). For supplementary compute-aligned plots only, opt in with `compare_group("localepochs", x_mode="cum_local_epochs")` (x = `r × local_epochs`).

**Communication cost:** `compare_strategy_nclients_communication()` draws macro-F1, accuracy, and loss vs. approximate cumulative client-slot uploads (strategy | client count).

**How to use**

1. Complete training ([`skinfl_colab_experiments`](skinfl_colab_experiments.ipynb)) so `results/<EXPERIMENT_KEY>/<run_name>/` exists (ideally with refreshed `summary.json` metadata; Setup sets `EXPERIMENT_KEY`). For ISIC 2018 runs, use [`skinfl_colab_visualizations_isic2018`](skinfl_colab_visualizations_isic2018.ipynb) instead.
2. Run **Setup** once below.
3. Run **Visualization imports** once.
4. Run single-run cells as needed (`visualize_single_run` writes global curves and, for federated runs with `round_*_clients.json`, client-level journal figures automatically).
5. Run comparison cells; when generated, extras live under `JOURNAL_EXPORT_DIR / "compare_communication/"` and `JOURNAL_EXPORT_DIR / "dirichlet_skew/"` (see Setup for the resolved path).

**Saved results (`EXPERIMENT_RESULTS_DIR / <run_name>/`):** besides figures, each run folder stores metrics JSON.

- `round_NNN.json` — global server evaluation after each round (scalar keys align with `summary.json` history entries).
- `round_NNN_clients.json` — federated simulation only. Shape `{"round": N, "clients": [...]}`. Each client row has `client_id`, `n_train`, plus `train_loss`, `n_local_train`, `n_local_test`, and nested `local_test` when that partition has a local holdout (same metric suite as global eval: `macro_f1`, `auroc`, `confusion_matrix`, etc.). Only clients participating **that round** appear.
- `summary.json` — concatenated `history`, optional `experiment` / `stopped`.

**Client-level journal exports:** when those files exist, `visualize_single_run` also writes line plots (round × metric, one trace per participating client): `18_lines_client_train_loss`, and (if `local_test` is non-empty) `19_lines_client_local_macro_f1` and `20_lines_client_local_auroc` under `JOURNAL_EXPORT_DIR / <run_name>/`. Centralized runs usually have no `round_*_clients.json`.

**Calibration:** reliability diagrams remain training-time PNGs (`figures/<run_name>/calibration_round_*.png`). This notebook lists them via `list_calibration_pngs`.

**Experiment parity:** sections mirror [`skinfl_colab_experiments`](skinfl_colab_experiments.ipynb).



In [ ]:
from __future__ import annotations

import logging
import os
import sys
from pathlib import Path

if sys.version_info < (3, 11):
    raise RuntimeError(
        f"This project requires Python >= 3.11 (pyproject.toml). Got: {sys.version}"
    )


def _in_colab() -> bool:
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except ImportError:
        return False


def _runtime_mode() -> str:
    # local = this machine; colab_drive = mount Drive and use COLAB_DRIVE_ROOT.
    env = os.environ.get("SKINFL_RUNTIME", "").strip().lower()
    if env in ("local", "colab_drive"):
        return env
    return "colab_drive" if _in_colab() else "local"


COLAB_DRIVE_ROOT = Path("/content/drive/MyDrive/SkinFL")
COLAB_SESSION_DATA_ROOT = Path("/content/skinfl_data")
PROJECT_ROOT_OVERRIDE: Path | None = None
DROP_MISSING_IMAGES: bool | None = None


def _discover_project_root() -> Path:
    start = Path.cwd().resolve()
    candidates = [start]
    if start.name == "notebooks":
        candidates.append(start.parent)
    for cand in candidates:
        if (cand / "pyproject.toml").is_file() and (cand / "src").is_dir():
            return cand
    for parent in start.parents:
        if (parent / "pyproject.toml").is_file() and (parent / "src").is_dir():
            return parent
    raise FileNotFoundError(
        "Could not find SkinFL repo (pyproject.toml + src/). "
        "cd to repo root or notebooks/, or set PROJECT_ROOT_OVERRIDE, "
        "or SKINFL_RUNTIME=colab_drive on Colab."
    )


RUNTIME = _runtime_mode()

if RUNTIME == "colab_drive":
    try:
        from google.colab import drive
    except ImportError as e:
        raise SystemExit(
            'SKINFL_RUNTIME=colab_drive but not in Colab. '
            "Unset SKINFL_RUNTIME or use local Jupyter."
        ) from e
    drive.mount("/content/drive")
    PROJECT_ROOT = COLAB_DRIVE_ROOT
else:
    PROJECT_ROOT = PROJECT_ROOT_OVERRIDE or _discover_project_root()

if RUNTIME == "colab_drive":
    DATA_ROOT = COLAB_SESSION_DATA_ROOT
else:
    DATA_ROOT = PROJECT_ROOT / "data"

RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
JOURNAL_OUT = FIGURES_DIR / "journal"

# One knob: must match the `results/<EXPERIMENT_KEY>/` tree (see experiments notebook / configs).
EXPERIMENT_KEY = "isic2019"
EXPERIMENT_RESULTS_DIR = RESULTS_DIR / EXPERIMENT_KEY
JOURNAL_EXPORT_DIR = JOURNAL_OUT / EXPERIMENT_KEY

DATA_ROOT.mkdir(parents=True, exist_ok=True)
for path in (RESULTS_DIR, FIGURES_DIR, JOURNAL_OUT, EXPERIMENT_RESULTS_DIR, JOURNAL_EXPORT_DIR):
    path.mkdir(parents=True, exist_ok=True)

if not (PROJECT_ROOT / "pyproject.toml").is_file():
    raise FileNotFoundError(
        f"Invalid PROJECT_ROOT: {PROJECT_ROOT} (pyproject.toml missing)."
    )

os.chdir(PROJECT_ROOT)

_repo_root = str(PROJECT_ROOT.resolve())
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")

_drop = (RUNTIME == "colab_drive") if DROP_MISSING_IMAGES is None else DROP_MISSING_IMAGES
if _drop:
    os.environ["SKINFL_DROP_MISSING_IMAGES"] = "1"
else:
    os.environ.pop("SKINFL_DROP_MISSING_IMAGES", None)

print("RUNTIME:", RUNTIME)
print("PROJECT_ROOT:", PROJECT_ROOT.resolve())
print("DATA_ROOT:", DATA_ROOT)
print("RESULTS_DIR:", RESULTS_DIR)
print("EXPERIMENT_KEY:", EXPERIMENT_KEY)
print("EXPERIMENT_RESULTS_DIR:", EXPERIMENT_RESULTS_DIR)
print("FIGURES_DIR:", FIGURES_DIR)
print("JOURNAL_OUT:", JOURNAL_OUT)
print("JOURNAL_EXPORT_DIR:", JOURNAL_EXPORT_DIR)
print(
    "SKINFL_DROP_MISSING_IMAGES:",
    os.environ.get("SKINFL_DROP_MISSING_IMAGES", "(unset — strict image checks)"),
)
if _drop:
    print(
        "Note: rows without a file on disk are dropped (subset run). "
        "Set DROP_MISSING_IMAGES=False after a full extract (official ISIC download cell) for strict counts."
    )


In [ ]:
# %pip install -q -e ".[dev]"


In [ ]:
from __future__ import annotations

from src.report.journal_viz import (
    client_class_counts_from_manifest,
    compare_group as _compare_group_impl,
    compare_strategy_nclients_communication as _compare_comm_impl,
    comparison_group_figure_title,
    get_clean_run_name,
    list_calibration_pngs as _list_cal_impl,
    plot_client_data_distribution,
    visualize_single_run as _viz_single_impl,
)


def visualize_single_run(run_name: str) -> None:
    """Write SVG + PDF figures under ``JOURNAL_EXPORT_DIR / run_name /``."""
    _viz_single_impl(run_name, results_dir=EXPERIMENT_RESULTS_DIR, journal_out=JOURNAL_EXPORT_DIR)


def compare_group(group_key: str, *, x_mode: str = "round") -> None:
    """Multi-run comparisons under ``JOURNAL_EXPORT_DIR / compare_<group>/``.

    Use ``x_mode="cum_local_epochs"`` only if you want cumulative local epochs on the x-axis (optional).
    """
    _compare_group_impl(
        group_key,
        results_dir=EXPERIMENT_RESULTS_DIR,
        journal_out=JOURNAL_EXPORT_DIR,
        project_root=PROJECT_ROOT,
        x_mode=x_mode,
    )


def compare_strategy_nclients_communication() -> None:
    """Macro-F1, accuracy, loss vs cumulative client-slot uploads (strategy | nclients)."""
    _compare_comm_impl(
        results_dir=EXPERIMENT_RESULTS_DIR,
        journal_out=JOURNAL_EXPORT_DIR,
        project_root=PROJECT_ROOT,
    )


def list_calibration_pngs(run_name: str) -> None:
    _list_cal_impl(run_name, figures_dir=FIGURES_DIR)


## Visualization imports (`src/report/journal_viz.py`)
After setup, run the cell above once. Sections below mirror [`skinfl_colab_experiments`](skinfl_colab_experiments.ipynb).


## Per-client JSON (quick check)

Federated runs write one `round_*_clients.json` per global round under `EXPERIMENT_RESULTS_DIR / <run_name>/`. **Centralized** training does not produce these files. Use the cell below to list them and inspect the payload shape (large nested lists in `local_test` are truncated for display).


In [ ]:
from __future__ import annotations

import json
from pprint import pprint

RUN_NAME = "isic2019_dirichlet"
run_dir = EXPERIMENT_RESULTS_DIR / RUN_NAME

paths = sorted(run_dir.glob("round_*_clients.json"))
print(f"{RUN_NAME}: {len(paths)} client JSON file(s)")
for p in paths[:15]:
    print(" ", p.name)
if len(paths) > 15:
    print(f"  ... ({len(paths) - 15} more)")

if not paths:
    print("No round_*_clients.json (expected for centralized runs or missing training output).")
else:
    first = paths[0]
    with first.open(encoding="utf-8") as fp:
        payload = json.load(fp)
    print("\nFirst file:", first.name)
    print("Top-level keys:", sorted(payload.keys()))
    clients = payload.get("clients") or []
    print("len(clients):", len(clients))

    def _truncate_client_row(row: dict, *, cm_row_max: int = 4, pcf_max: int = 12) -> dict:
        out = dict(row)
        lt = out.get("local_test")
        if isinstance(lt, dict):
            lt = dict(lt)
            pcm = lt.get("per_class_f1")
            if isinstance(pcm, list) and len(pcm) > pcf_max:
                lt["per_class_f1"] = pcm[:pcf_max] + [f"... ({len(pcm) - pcf_max} more classes)"]
            cm = lt.get("confusion_matrix")
            if isinstance(cm, list) and len(cm) > cm_row_max:
                lt["confusion_matrix"] = cm[:cm_row_max] + [
                    [f"... ({len(cm) - cm_row_max} more rows)"]
                ]
            out["local_test"] = lt
        return out

    if clients:
        pprint(_truncate_client_row(clients[0]))
    else:
        print("(empty clients list)")


## Primary experiments

Main ISIC2019 FL (`isic2019_dirichlet`) and pooled centralized upper bound (same as the experiments notebook).

### Single run: `isic2019_dirichlet`

Run the code cell below for this experiment only.


In [ ]:
RUN_NAME = "isic2019_dirichlet"
visualize_single_run(RUN_NAME)
# list_calibration_pngs(RUN_NAME)  # optional


### Single run: `centralized_upperbound`

Run the code cell below for this experiment only.


In [ ]:
RUN_NAME = "centralized_upperbound"
visualize_single_run(RUN_NAME)
# list_calibration_pngs(RUN_NAME)  # optional


## Partition ablations (ISIC2019, Dirichlet alpha sweep)

Alpha values 0.1, 0.5, and 1.0. The headline `isic2019_dirichlet` run uses its own YAML (see experiments notebook).

### Single run: `ablation_partition_dirichlet_01`

Run the code cell below for this experiment only.


In [ ]:
RUN_NAME = "ablation_partition_dirichlet_01"
visualize_single_run(RUN_NAME)
# list_calibration_pngs(RUN_NAME)  # optional


### Single run: `ablation_partition_dirichlet_05`

Run the code cell below for this experiment only.


In [ ]:
RUN_NAME = "ablation_partition_dirichlet_05"
visualize_single_run(RUN_NAME)
# list_calibration_pngs(RUN_NAME)  # optional


### Single run: `ablation_partition_dirichlet_10`

Run the code cell below for this experiment only.


In [ ]:
RUN_NAME = "ablation_partition_dirichlet_10"
visualize_single_run(RUN_NAME)
# list_calibration_pngs(RUN_NAME)  # optional


## Optional: IID / patient partition

Uncomment in the experiments notebook if you ran these configs.

### Single run: `ablation_partition_iid`

Run the code cell below for this experiment only.


In [ ]:
RUN_NAME = "ablation_partition_iid"
visualize_single_run(RUN_NAME)
# list_calibration_pngs(RUN_NAME)  # optional


### Single run: `ablation_partition_patient`

Run the code cell below for this experiment only.


In [ ]:
RUN_NAME = "ablation_partition_patient"
visualize_single_run(RUN_NAME)
# list_calibration_pngs(RUN_NAME)  # optional


## Client-count ablations

### Single run: `ablation_nclients_5`

Run the code cell below for this experiment only.


In [ ]:
RUN_NAME = "ablation_nclients_5"
visualize_single_run(RUN_NAME)
# list_calibration_pngs(RUN_NAME)  # optional


### Single run: `ablation_nclients_10`

Run the code cell below for this experiment only.


In [ ]:
RUN_NAME = "ablation_nclients_10"
visualize_single_run(RUN_NAME)
# list_calibration_pngs(RUN_NAME)  # optional


### Single run: `ablation_nclients_20`

Run the code cell below for this experiment only.


In [ ]:
RUN_NAME = "ablation_nclients_20"
visualize_single_run(RUN_NAME)
# list_calibration_pngs(RUN_NAME)  # optional


## Strategy ablations

### Single run: `ablation_strategy_fedprox`

Run the code cell below for this experiment only.


In [ ]:
RUN_NAME = "ablation_strategy_fedprox"
visualize_single_run(RUN_NAME)
# list_calibration_pngs(RUN_NAME)  # optional


### Single run: `ablation_strategy_fedadam`

Run the code cell below for this experiment only.


In [ ]:
RUN_NAME = "ablation_strategy_fedadam"
visualize_single_run(RUN_NAME)
# list_calibration_pngs(RUN_NAME)  # optional


## Local-epoch ablations

### Single run: `ablation_localepochs_1`

Run the code cell below for this experiment only.


In [ ]:
RUN_NAME = "ablation_localepochs_1"
visualize_single_run(RUN_NAME)
# list_calibration_pngs(RUN_NAME)  # optional


### Single run: `ablation_localepochs_5`

Run the code cell below for this experiment only.


In [ ]:
RUN_NAME = "ablation_localepochs_5"
visualize_single_run(RUN_NAME)
# list_calibration_pngs(RUN_NAME)  # optional


### Single run: `ablation_localepochs_10`

Run the code cell below for this experiment only.


In [ ]:
RUN_NAME = "ablation_localepochs_10"
visualize_single_run(RUN_NAME)
# list_calibration_pngs(RUN_NAME)  # optional


## Cross-run comparisons

Outputs go to `JOURNAL_EXPORT_DIR / "compare_<group>/"` (on disk: `figures/journal/<EXPERIMENT_KEY>/compare_<group>/`; SVG + PDF). Keys: `partition`, `nclients`, `strategy`, `localepochs`, `primary`.

For **`localepochs`**, the default comparison uses **Round** on the x-axis (consistent with other groups).

**`primary`** compares ISIC2019 FL (`isic2019_dirichlet`) vs centralized pooled baseline.

Combined **strategy + client-count vs communication cost** lives in `JOURNAL_EXPORT_DIR / "compare_communication/"` (run the dedicated cell below).



### Comparison group: `partition`


In [ ]:
compare_group("partition")


### Comparison group: `nclients`


In [ ]:
compare_group("nclients")


### Comparison group: `strategy`


In [ ]:
compare_group("strategy")


### Comparison group: `localepochs`


In [ ]:
compare_group("localepochs")


### Combined: strategy and client-count vs communication cost

Approximate x-axis: cumulative client-slot uploads per federated protocol (`max(1, int(num_clients * fraction_fit))` per global round \(r \ge 1\)). Centralized runs contribute flat \(x=0\) if mixed in accidentally — prefer FL-only lists.


In [ ]:
compare_strategy_nclients_communication()


### Dirichlet partition skew (optional heatmap)

Builds a client × class count matrix from a **post-partition** `manifest.csv` (`train` rows with `client_id` and `label`). Each experiment run overwrites `DATA_ROOT/ISIC2019/manifest.csv`; **save CSV snapshots** under different filenames before comparing \(\alpha=100\) vs \(\alpha=0.1\).


In [ ]:
# Example: current manifest only — set alpha_val to match the partition you trained with.

manifest_path = DATA_ROOT / "ISIC2019" / "manifest.csv"
counts, classes = client_class_counts_from_manifest(manifest_path)
plot_client_data_distribution(
    counts,
    alpha_val=100.0,  # e.g. 100.0 or 0.1; use None if unknown
    class_names=classes,
    out_path_without_ext=JOURNAL_EXPORT_DIR / "dirichlet_skew" / "manifest_current",
)


### Comparison group: `primary`


In [ ]:
compare_group("primary")
